# Full Auto-Tuning Run For Uplift Leaderboard

Боевой ноутбук для попытки залезть выше в leaderboard.

Что делает:

- подбирает гиперпараметры **не по RMSE/AUC**, а по `uplift@10` и `bootstrap_lower_80`;
- сначала гоняет random search для `Hurdle T-learner`;
- перепроверяет лучшие конфиги на stress-holdout по верхним `user_id`;
- обучает multi-seed rank ensemble на полном train;
- генерирует несколько submission-кандидатов, включая per-communication модель.

Главная идея: на дистанции лучше не одна удачная модель, а устойчивый ensemble ранжирований.

In [ ]:
import gc
import json
import os
import random
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor
from IPython.display import display
from sklearn.model_selection import StratifiedShuffleSplit

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

DATA_DIR = Path("кей")
TRAIN_PATH = DATA_DIR / "train.parquet"
TEST_PATH = DATA_DIR / "test.parquet"

ID_COL = "user_id"
TREATMENT_COL = "treatment_flg"
TARGET_COL = "rec_spend"
COMM_COL = "communication_type"

TOP_K = 0.10
BOOTSTRAP_ITERS = 200
BOOTSTRAP_LOWER_Q = 0.10

VALID_SIZE = 0.25
THREAD_COUNT = 4

# Full run knobs. If time is tight, reduce HPO_TRIALS first.
HPO_TRIALS = 40
TOP_CONFIGS_FOR_STRESS = 8
FINAL_TOP_CONFIGS = 5
FINAL_SEEDS = [42, 777, 2026]

RUN_HPO = True
RUN_STRESS_RECHECK = True
FIT_FINAL_ENSEMBLES = True
GENERATE_PER_COMM_CANDIDATE = True

MAIN_SUBMISSION = "auto_blend_hurdle_percomm_s"

ARTIFACT_DIR = Path("auto_tuning_artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)
HPO_RESULTS_PATH = ARTIFACT_DIR / "hpo_results_hurdle.csv"
STRESS_RESULTS_PATH = ARTIFACT_DIR / "hpo_stress_recheck.csv"

## Load Data

In [ ]:
train = pd.read_parquet(TRAIN_PATH)
test = pd.read_parquet(TEST_PATH)

FEATURES = [c for c in test.columns if c != ID_COL]
CAT_FEATURES = [COMM_COL]
FEATURES_NO_COMM = [c for c in FEATURES if c != COMM_COL]
STRATA = train[TREATMENT_COL].astype(str) + "_" + train[COMM_COL].astype(str)

print("train:", train.shape)
print("test :", test.shape)
print("features:", len(FEATURES))
print("user_id order overlap:", len(set(train[ID_COL]) & set(test[ID_COL])))
print("target zero share:", round((train[TARGET_COL] == 0).mean(), 4))
display(pd.crosstab(train[COMM_COL], train[TREATMENT_COL], normalize="index"))

## Metric

In [ ]:
def uplift_at_k(y_true, treatment, score, k=TOP_K):
    y_true = np.asarray(y_true, dtype=float)
    treatment = np.asarray(treatment, dtype=int)
    score = np.asarray(score, dtype=float)

    n_top = max(1, int(len(score) * k))
    top_idx = np.argsort(-score, kind="mergesort")[:n_top]
    top_y = y_true[top_idx]
    top_w = treatment[top_idx]
    treat_mask = top_w == 1
    control_mask = top_w == 0

    if treat_mask.sum() == 0 or control_mask.sum() == 0:
        uplift = np.nan
    else:
        uplift = top_y[treat_mask].mean() - top_y[control_mask].mean()

    return {
        "uplift_at_k": float(uplift),
        "n_total": int(len(score)),
        "n_top": int(n_top),
        "top_treatment_n": int(treat_mask.sum()),
        "top_control_n": int(control_mask.sum()),
        "top_target_mean": float(top_y.mean()),
        "top_score_mean": float(score[top_idx].mean()),
    }


def bootstrap_uplift_at_k(
    y_true,
    treatment,
    score,
    k=TOP_K,
    n_bootstrap=BOOTSTRAP_ITERS,
    lower_q=BOOTSTRAP_LOWER_Q,
    random_state=RANDOM_STATE,
):
    y_true = np.asarray(y_true, dtype=float)
    treatment = np.asarray(treatment, dtype=int)
    score = np.asarray(score, dtype=float)

    base = uplift_at_k(y_true, treatment, score, k=k)
    n_top = base["n_top"]
    top_idx = np.argsort(-score, kind="mergesort")[:n_top]
    top_y = y_true[top_idx]
    top_w = treatment[top_idx]

    rng = np.random.default_rng(random_state)
    boot_values = []
    for _ in range(n_bootstrap):
        sample_idx = rng.integers(0, n_top, size=n_top)
        sample_y = top_y[sample_idx]
        sample_w = top_w[sample_idx]
        treat_mask = sample_w == 1
        control_mask = sample_w == 0
        if treat_mask.sum() == 0 or control_mask.sum() == 0:
            continue
        boot_values.append(sample_y[treat_mask].mean() - sample_y[control_mask].mean())

    boot_values = np.asarray(boot_values, dtype=float)
    result = dict(base)
    result.update(
        {
            "bootstrap_mean": float(np.mean(boot_values)),
            "bootstrap_std": float(np.std(boot_values)),
            "bootstrap_lower_80": float(np.quantile(boot_values, lower_q)),
            "bootstrap_upper_80": float(np.quantile(boot_values, 1 - lower_q)),
            "bootstrap_iters_used": int(len(boot_values)),
        }
    )
    return result


def robust_objective(metrics):
    # Lower bound matters because the official metric bootstraps top-10%.
    # Raw uplift still gets a small weight to avoid overly conservative candidates.
    return 0.75 * metrics["bootstrap_lower_80"] + 0.25 * metrics["uplift_at_k"]

## Model Helpers

In [ ]:
def catboost_common_params(cfg, seed, verbose=False):
    params = {
        "iterations": int(cfg["iterations"]),
        "depth": int(cfg["depth"]),
        "learning_rate": float(cfg["learning_rate"]),
        "l2_leaf_reg": float(cfg["l2_leaf_reg"]),
        "random_strength": float(cfg["random_strength"]),
        "rsm": float(cfg["rsm"]),
        "random_seed": int(seed),
        "thread_count": THREAD_COUNT,
        "verbose": verbose,
        "allow_writing_files": False,
    }

    bootstrap_type = cfg.get("bootstrap_type", "Bayesian")
    params["bootstrap_type"] = bootstrap_type
    if bootstrap_type == "Bayesian":
        params["bagging_temperature"] = float(cfg.get("bagging_temperature", 1.0))
    elif bootstrap_type in {"Bernoulli", "MVS"}:
        params["subsample"] = float(cfg.get("subsample", 0.8))

    return params


def make_prob_params(cfg, seed, verbose=False):
    params = catboost_common_params(cfg, seed, verbose=verbose)
    params.update({"loss_function": "Logloss", "eval_metric": "AUC"})
    if cfg.get("auto_class_weights") is not None:
        params["auto_class_weights"] = cfg["auto_class_weights"]
    return params


def make_amount_params(cfg, seed, verbose=False):
    params = catboost_common_params(cfg, seed, verbose=verbose)
    loss = cfg.get("amount_loss", "RMSE")
    params.update({"loss_function": loss, "eval_metric": loss})
    return params


def make_response_params(cfg, seed, verbose=False):
    params = catboost_common_params(cfg, seed, verbose=verbose)
    params.update({"loss_function": "RMSE", "eval_metric": "RMSE"})
    return params


def fit_hurdle_t_learner(
    df,
    cfg,
    features=FEATURES,
    cat_features=CAT_FEATURES,
    seed=RANDOM_STATE,
    verbose=False,
):
    models = {}

    for arm in [0, 1]:
        arm_mask = df[TREATMENT_COL].eq(arm)
        x_arm = df.loc[arm_mask, features]
        y_arm = df.loc[arm_mask, TARGET_COL].astype(float)
        y_nonzero = y_arm.gt(0).astype(int)

        prob_model = CatBoostClassifier(**make_prob_params(cfg, seed=seed + arm, verbose=verbose))
        prob_model.fit(x_arm, y_nonzero, cat_features=cat_features)

        positive_mask = y_arm.gt(0)
        amount_model = CatBoostRegressor(**make_amount_params(cfg, seed=seed + 10 + arm, verbose=verbose))
        amount_target = np.log1p(y_arm.loc[positive_mask])
        amount_model.fit(x_arm.loc[positive_mask], amount_target, cat_features=cat_features)

        models[arm] = {
            "prob_model": prob_model,
            "amount_model": amount_model,
            "n_rows": int(len(y_arm)),
            "n_positive": int(positive_mask.sum()),
        }

    return models


def predict_expected_spend(models, x, arm):
    prob = models[arm]["prob_model"].predict_proba(x)[:, 1]
    amount = np.expm1(models[arm]["amount_model"].predict(x))
    amount = np.clip(amount, 0, None)
    return prob * amount


def predict_hurdle_uplift(models, x):
    return predict_expected_spend(models, x, 1) - predict_expected_spend(models, x, 0)


def fit_s_learner(df, cfg, features=FEATURES, cat_features=CAT_FEATURES, seed=RANDOM_STATE):
    x = df[features].copy()
    x[TREATMENT_COL] = df[TREATMENT_COL].astype(int).values
    model = CatBoostRegressor(**make_response_params(cfg, seed=seed))
    model.fit(x, df[TARGET_COL].astype(float).values, cat_features=cat_features)
    return model


def predict_s_learner(model, x, features=FEATURES):
    x_treat = x[features].copy()
    x_control = x[features].copy()
    x_treat[TREATMENT_COL] = 1
    x_control[TREATMENT_COL] = 0
    return model.predict(x_treat) - model.predict(x_control)


def fit_transformed_outcome_model(df, cfg, features=FEATURES, cat_features=CAT_FEATURES, seed=RANDOM_STATE):
    p_treat = df[TREATMENT_COL].mean()
    pseudo_uplift = df[TARGET_COL].astype(float).values * (df[TREATMENT_COL].values - p_treat) / (p_treat * (1 - p_treat))
    model = CatBoostRegressor(**make_response_params(cfg, seed=seed))
    model.fit(df[features], pseudo_uplift, cat_features=cat_features)
    return model


def rank_average(*score_arrays):
    ranks = [pd.Series(scores).rank(method="average", ascending=True).to_numpy() for scores in score_arrays]
    return np.mean(ranks, axis=0)


def save_submission(scores, output_path, test_df=test):
    submission = pd.DataFrame({ID_COL: test_df[ID_COL].values, "UPLIFT_SCORE": scores})
    assert len(submission) == len(test_df)
    assert submission[ID_COL].tolist() == test_df[ID_COL].tolist()
    assert submission["UPLIFT_SCORE"].notna().all()
    assert np.isfinite(submission["UPLIFT_SCORE"]).all()
    submission.to_csv(output_path, index=False)
    return submission

## Validation Splits

In [ ]:
def make_stratified_holdout(df=train):
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=VALID_SIZE, random_state=RANDOM_STATE)
    tr_idx, va_idx = next(splitter.split(df, STRATA))
    return df.iloc[tr_idx].reset_index(drop=True), df.iloc[va_idx].reset_index(drop=True)


def make_upper_user_id_holdout(df=train):
    ordered_idx = df.sort_values(ID_COL).index.to_numpy()
    cut = int(len(ordered_idx) * (1 - VALID_SIZE))
    return df.loc[ordered_idx[:cut]].reset_index(drop=True), df.loc[ordered_idx[cut:]].reset_index(drop=True)


train_strat, valid_strat = make_stratified_holdout()
train_upper, valid_upper = make_upper_user_id_holdout()

print("strat holdout:", train_strat.shape, valid_strat.shape)
print("upper holdout:", train_upper.shape, valid_upper.shape)

## Random Search Space

In [ ]:
def sample_hurdle_config(rng, trial_id):
    bootstrap_type = rng.choice(["Bayesian", "Bernoulli", "MVS"], p=[0.45, 0.35, 0.20])
    cfg = {
        "trial_id": int(trial_id),
        "iterations": int(rng.choice([500, 700, 900, 1200, 1600])),
        "depth": int(rng.choice([4, 5, 6, 7])),
        "learning_rate": float(np.round(np.exp(rng.uniform(np.log(0.025), np.log(0.075))), 4)),
        "l2_leaf_reg": float(np.round(np.exp(rng.uniform(np.log(1.0), np.log(25.0))), 4)),
        "random_strength": float(rng.choice([0.0, 0.5, 1.0, 2.0, 4.0])),
        "rsm": float(rng.choice([0.75, 0.85, 0.95, 1.0])),
        "bootstrap_type": str(bootstrap_type),
        "amount_loss": str(rng.choice(["RMSE", "MAE"], p=[0.75, 0.25])),
        "auto_class_weights": rng.choice([None, "Balanced", "SqrtBalanced"], p=[0.65, 0.15, 0.20]),
    }
    if cfg["auto_class_weights"] is not None:
        cfg["auto_class_weights"] = str(cfg["auto_class_weights"])
    if bootstrap_type == "Bayesian":
        cfg["bagging_temperature"] = float(np.round(rng.uniform(0.0, 2.5), 3))
    else:
        cfg["subsample"] = float(np.round(rng.uniform(0.65, 0.95), 3))
    return cfg


def config_to_json(cfg):
    return json.dumps(cfg, sort_keys=True, ensure_ascii=False)


def evaluate_hurdle_config(train_part, valid_part, cfg, seed):
    models = fit_hurdle_t_learner(train_part, cfg, seed=seed)
    scores = predict_hurdle_uplift(models, valid_part[FEATURES])
    metrics = bootstrap_uplift_at_k(
        valid_part[TARGET_COL].values,
        valid_part[TREATMENT_COL].values,
        scores,
        random_state=seed,
    )
    metrics["objective"] = robust_objective(metrics)
    return metrics

## HPO: Optimize Hurdle T-Learner On Stratified Holdout

In [ ]:
if RUN_HPO:
    rng = np.random.default_rng(RANDOM_STATE)
    trial_rows = []

    for trial_id in range(1, HPO_TRIALS + 1):
        cfg = sample_hurdle_config(rng, trial_id)
        print("=" * 90)
        print(f"trial {trial_id}/{HPO_TRIALS}")
        print(config_to_json(cfg))

        t0 = time.time()
        try:
            metrics = evaluate_hurdle_config(
                train_strat,
                valid_strat,
                cfg,
                seed=RANDOM_STATE + 1000 + trial_id,
            )
            row = {
                **cfg,
                **metrics,
                "config_json": config_to_json(cfg),
                "elapsed_min": (time.time() - t0) / 60,
                "status": "ok",
            }
        except Exception as exc:
            row = {
                **cfg,
                "config_json": config_to_json(cfg),
                "elapsed_min": (time.time() - t0) / 60,
                "status": "fail",
                "error": repr(exc),
            }
            print("FAILED:", repr(exc))

        trial_rows.append(row)
        pd.DataFrame(trial_rows).to_csv(HPO_RESULTS_PATH, index=False)

        if row["status"] == "ok":
            print(
                f"uplift={row['uplift_at_k']:.4f} "
                f"lower80={row['bootstrap_lower_80']:.4f} "
                f"objective={row['objective']:.4f} "
                f"elapsed={row['elapsed_min']:.1f}m"
            )

        gc.collect()

    hpo_results = pd.DataFrame(trial_rows)
else:
    hpo_results = pd.read_csv(HPO_RESULTS_PATH)

hpo_ok = hpo_results[hpo_results["status"].eq("ok")].copy()
hpo_ok = hpo_ok.sort_values(["objective", "bootstrap_lower_80", "uplift_at_k"], ascending=False)
display(hpo_ok.head(15))

## Stress Recheck Best Configs

In [ ]:
def parse_config_row(row):
    cfg = json.loads(row["config_json"])
    return cfg


if RUN_STRESS_RECHECK:
    top_for_stress = hpo_ok.head(TOP_CONFIGS_FOR_STRESS).copy()
    stress_rows = []

    for rank, (_, row) in enumerate(top_for_stress.iterrows(), start=1):
        cfg = parse_config_row(row)
        print("=" * 90)
        print(f"stress recheck {rank}/{len(top_for_stress)} trial_id={cfg['trial_id']}")

        t0 = time.time()
        metrics = evaluate_hurdle_config(
            train_upper,
            valid_upper,
            cfg,
            seed=RANDOM_STATE + 5000 + int(cfg["trial_id"]),
        )
        stress_row = {
            **cfg,
            **metrics,
            "config_json": config_to_json(cfg),
            "elapsed_min": (time.time() - t0) / 60,
        }
        stress_rows.append(stress_row)
        pd.DataFrame(stress_rows).to_csv(STRESS_RESULTS_PATH, index=False)
        print(
            f"stress uplift={metrics['uplift_at_k']:.4f} "
            f"lower80={metrics['bootstrap_lower_80']:.4f} "
            f"objective={metrics['objective']:.4f}"
        )
        gc.collect()

    stress_results = pd.DataFrame(stress_rows)
else:
    stress_results = pd.read_csv(STRESS_RESULTS_PATH)

keep_cols = ["trial_id", "objective", "bootstrap_lower_80", "uplift_at_k", "config_json"]
strat_summary = hpo_ok[keep_cols].rename(
    columns={
        "objective": "strat_objective",
        "bootstrap_lower_80": "strat_lower80",
        "uplift_at_k": "strat_uplift",
    }
)
stress_summary = stress_results[keep_cols].rename(
    columns={
        "objective": "stress_objective",
        "bootstrap_lower_80": "stress_lower80",
        "uplift_at_k": "stress_uplift",
    }
)
config_scores = strat_summary.merge(stress_summary, on=["trial_id", "config_json"], how="inner")
config_scores["combined_objective"] = (
    0.55 * config_scores["strat_objective"]
    + 0.35 * config_scores["stress_objective"]
    + 0.10 * np.minimum(config_scores["strat_lower80"], config_scores["stress_lower80"])
)
config_scores = config_scores.sort_values("combined_objective", ascending=False)
display(config_scores)

best_configs = [json.loads(x) for x in config_scores.head(FINAL_TOP_CONFIGS)["config_json"]]
print("selected trial_ids:", [cfg["trial_id"] for cfg in best_configs])

## Final Multi-Seed Ensembles

In [ ]:
def fit_predict_full_hurdle_ensemble(configs, seeds, label):
    scores = []
    meta_rows = []

    for cfg_rank, cfg in enumerate(configs, start=1):
        for seed in seeds:
            print("=" * 90)
            print(f"fit full {label}: cfg_rank={cfg_rank}, trial_id={cfg['trial_id']}, seed={seed}")
            t0 = time.time()
            models = fit_hurdle_t_learner(train, cfg, seed=seed)
            pred = predict_hurdle_uplift(models, test[FEATURES])
            scores.append(pred)
            meta_rows.append(
                {
                    "label": label,
                    "cfg_rank": cfg_rank,
                    "trial_id": cfg["trial_id"],
                    "seed": seed,
                    "elapsed_min": (time.time() - t0) / 60,
                    "score_mean": float(np.mean(pred)),
                    "score_std": float(np.std(pred)),
                }
            )
            gc.collect()

    return scores, pd.DataFrame(meta_rows)


def fit_predict_per_comm_hurdle(cfg, seed=RANDOM_STATE):
    scores = np.zeros(len(test), dtype=float)
    meta_rows = []

    for comm in sorted(test[COMM_COL].dropna().unique()):
        train_part = train[train[COMM_COL].eq(comm)].reset_index(drop=True)
        test_mask = test[COMM_COL].eq(comm).to_numpy()
        test_part = test.loc[test_mask].reset_index(drop=True)
        print("=" * 90)
        print(f"fit per-communication hurdle: {comm}, train={train_part.shape}, test={test_part.shape}")
        t0 = time.time()
        models = fit_hurdle_t_learner(
            train_part,
            cfg,
            features=FEATURES_NO_COMM,
            cat_features=[],
            seed=seed + sum(ord(ch) for ch in str(comm)),
        )
        scores[test_mask] = predict_hurdle_uplift(models, test_part[FEATURES_NO_COMM])
        meta_rows.append({"communication_type": comm, "elapsed_min": (time.time() - t0) / 60})
        gc.collect()

    return scores, pd.DataFrame(meta_rows)


if FIT_FINAL_ENSEMBLES:
    final_scores = {}
    final_meta = []

    hurdle_member_scores, hurdle_meta = fit_predict_full_hurdle_ensemble(
        best_configs,
        FINAL_SEEDS,
        label="auto_hurdle",
    )
    final_meta.append(hurdle_meta)

    # Best single full hurdle member and rank ensemble over all HPO members.
    final_scores["auto_best_hurdle_single"] = hurdle_member_scores[0]
    final_scores["auto_hurdle_rank_ensemble"] = rank_average(*hurdle_member_scores)

    best_cfg = best_configs[0]

    print("=" * 90)
    print("fit final S-learner on best config")
    s_model = fit_s_learner(train, best_cfg, seed=RANDOM_STATE + 8000)
    final_scores["auto_s_learner"] = predict_s_learner(s_model, test)

    print("=" * 90)
    print("fit final transformed outcome on best config")
    to_model = fit_transformed_outcome_model(train, best_cfg, seed=RANDOM_STATE + 8100)
    final_scores["auto_transformed_outcome"] = to_model.predict(test[FEATURES])

    if GENERATE_PER_COMM_CANDIDATE:
        per_comm_scores, per_comm_meta = fit_predict_per_comm_hurdle(best_cfg, seed=RANDOM_STATE + 8200)
        final_scores["auto_per_comm_hurdle"] = per_comm_scores
        final_meta.append(per_comm_meta)

    final_scores["auto_blend_hurdle_s"] = rank_average(
        final_scores["auto_hurdle_rank_ensemble"],
        final_scores["auto_s_learner"],
    )
    if "auto_per_comm_hurdle" in final_scores:
        final_scores["auto_blend_hurdle_percomm_s"] = rank_average(
            final_scores["auto_hurdle_rank_ensemble"],
            final_scores["auto_per_comm_hurdle"],
            final_scores["auto_s_learner"],
        )
    base_blend_keys = [
        "auto_hurdle_rank_ensemble",
        "auto_s_learner",
        "auto_transformed_outcome",
    ]
    if "auto_per_comm_hurdle" in final_scores:
        base_blend_keys.append("auto_per_comm_hurdle")
    final_scores["auto_blend_all"] = rank_average(*(final_scores[key] for key in base_blend_keys))

    submission_rows = []
    for name, scores in final_scores.items():
        path = Path(f"predictions_{name}.csv")
        save_submission(scores, path)
        submission_rows.append(
            {
                "submission": name,
                "path": str(path),
                "score_mean": float(np.mean(scores)),
                "score_std": float(np.std(scores)),
                "score_min": float(np.min(scores)),
                "score_max": float(np.max(scores)),
            }
        )

    main_scores = final_scores[MAIN_SUBMISSION]
    main_submission = save_submission(main_scores, "predictions.csv")
    print(f"main submission: {MAIN_SUBMISSION} -> predictions.csv")
    display(pd.DataFrame(submission_rows))
    display(main_submission.head())
    display(main_submission["UPLIFT_SCORE"].describe().to_frame("UPLIFT_SCORE"))

    if final_meta:
        final_meta_df = pd.concat(final_meta, ignore_index=True, sort=False)
        final_meta_df.to_csv(ARTIFACT_DIR / "final_fit_meta.csv", index=False)
        display(final_meta_df)

## Submission Order

Я бы отправлял в таком порядке:

1. `predictions.csv` / `predictions_auto_blend_hurdle_percomm_s.csv`
2. `predictions_auto_hurdle_rank_ensemble.csv`
3. `predictions_auto_blend_hurdle_s.csv`
4. `predictions_auto_per_comm_hurdle.csv`
5. `predictions_auto_best_hurdle_single.csv`

Если leaderboard ухудшился у per-communication blend, быстро переключи:

```python
MAIN_SUBMISSION = "auto_hurdle_rank_ensemble"
```

и перезапусти только финальную ячейку генерации файлов, если `final_scores` ещё в памяти.